# Notebook — Génération (Phases 1-3)

Pipeline TAF → X-Plane 12 → IoU. Équivaut à `run_pipeline.py generate / render / evaluate`.
Les fonctionnalités annexes (dataset, regroup, sanity, vidéo…) sont dans `notebook_features.ipynb`.

## 1. Setup

In [2]:
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "run_pipeline.py").exists():
            return p
    return start

ROOT = _find_root(Path.cwd())

_project = str(ROOT / "project")
if _project not in sys.path:
    sys.path.insert(0, _project)

print(f"ROOT = {ROOT}")

ROOT = c:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF


## 2. Generate (Phase 1 — TAF seul, offline)

Équivalent CLI : `py run_pipeline.py generate -n N`

Sample N scénarios via TAF/z3 et écrit `runs/<generation>/<ICAO_RWY>/` avec `.yaml`,
`poses_cam_export.json`, `fault_profile.json`, `weather_profile.json`.
Pas besoin de X-Plane. 
Si `nb_scenarios=None` => valeur de `project/settings.xml`.

In [4]:
from Generate import generate_runs

# generate_runs(nb_scenarios=5)
generate_runs(nb_scenarios=2);

 PHASE 1 : Generation TAF
[Pipeline] Generation : generation_01/
[Generate] Template : templates/base_template.xml
[Generate] Sortie   : ../output/
[Generate] Scenarios: 2

parsing template: templates/base_template.xml
Current path: ../output/scenario_0/
Miscellaneous::file_check() -> "../output/scenario_0/scenario_0.xml" does not exist
Create new test case: scenario.xml
--------------------------------------------------------------------------------
additional contraint(s): 
['var_533_0 == 282.74665', 'var_532_0 == 2000.0', 'var_538_0 == 1339.98586']
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
additional contraint(s): 
['var_469_0 == 100.0', 'var_457_0 == 100.0', 'var_478_0 == 100.0', 'var_475_0 == 100.0', 'var_453_0 == 0.0', 'var_465_0 == 1', 'var_493_0 == 100.0', 'var_462_0 == 10.0', 'var_472_0 == 100.0', 'var_464_0 == 2', 'var_504_0 == 0.0', 'var_459_0 == 0.0', 'var

## 3. Render (Phase 2 — Rendu X-Plane + fautes capteur)

Équivalent CLI :
- `py run_pipeline.py render generation_01/LFPO_25`
- `py run_pipeline.py render --all --generation generation_01`

Supposé que la Phase 1 (`generate`) a déjà été faite. Nécessite X-Plane 12
lancé (mode fenêtre, scaling 100%). Sortie : `footage/`, `degraded/`.


In [5]:
import xml.etree.ElementTree as ET
from runs import render_runs

# Répertoire X-Plane lu depuis project/settings.xml -> Chemin du simulateur
_settings = {p.attrib["name"]: p.attrib["value"]
             for p in ET.parse(ROOT / "project" / "settings.xml").getroot()}
XPLANE_DIR = _settings["xplane_dir"]

# Un seul run
# render_runs(run_name="generation_01/KLAX_25R", xplane_dir=XPLANE_DIR)

# Tous les runs :
render_runs(all_runs=True, xplane_dir=XPLANE_DIR);

 PHASE 2 : Rendu X-Plane + fautes capteur

[Pipeline] 2 run(s) a rendre

--------------------------------------------------
 Run : generation_01/KATL_8L
--------------------------------------------------

  [Image] Rendu + fautes + GT pour KATL_8L

  [XPLANE] Rendu de KATL_8L...
  [XPLANE] Rendu de KATL_8L (276 frames)...
  [XPLANE] Connexion UDP -> 127.0.0.1:49000
  [XPLANE] Fenetre resizee -> 1024x1024
  [XPLANE] Fenetre trouvee: 'X-Plane' rect=(276, 50, 1572, 1113)
  [XPLANE] Client rect: left=284 top=81 right=1564 bottom=1105
  [XPLANE] Desktop virtuel: {'left': 0, 'top': 0, 'width': 4480, 'height': 1440}
  [XPLANE] Capture : 1280x1024
  [XPLANE] Fenetre reelle: 1280x1024, fact=1.25x1.00
  [XPLANE] FOV programme: H=71.6° V=60.0° (readback: H=71.6° V=60.0°)
  [XPLANE] Plugin XPPython3 weather OK
  [WEATHER] Injecte : clouds=Cirrus(44%), vis=20243m, avion_max=425m, nuages=1227-4113m MSL
  [WEATHER] Chargement textures + stabilisation nuages 15.0s...
  [WEATHER] Stabilisation terminee

## 4. Evaluate (Phase 3 — GT LARD + YOLO + IoU, sans X-Plane)

Équivalent CLI : `py run_pipeline.py evaluate --all`.

Suppose que `footage/` ou `degraded/` existe déjà (Phase 2 faite). YOLO predict + IoU. Ré-exécutable à volonté
avec d'autres poids/seuils sans re-rendre.

In [6]:
from runs import evaluate_runs

# Un seul run (nom seul si unique, sinon chemin composé '<gen>/<run>') :
# evaluate_runs(run_name="generation_01/KLAX_25R")

# Tous les runs :
evaluate_runs(all_runs=True);

 PHASE 3 : Detection YOLO + IoU

[Pipeline] 2 run(s) a evaluer

--------------------------------------------------
 Run : generation_01/KATL_8L
--------------------------------------------------

  [Eval] Detection + IoU pour KATL_8L

  [YOLO] Prediction sur 276 images depuis footage/ (KATL_8L)...
Prediction sur 276 images avec yolov8nTest.pt

image 1/276 C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\generation_01\KATL_8L\footage\KATL_8L_000.jpg: 512x512 1 runway, 11.2ms
image 2/276 C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\generation_01\KATL_8L\footage\KATL_8L_001.jpg: 512x512 1 runway, 10.0ms
image 3/276 C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\generation_01\KATL_8L\footage\KATL_8L_002.jpg: 512x512 1 runway, 11.2ms
image 4/276 C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\generation_01\KATL_8L\footage\KATL_8L_003.jpg: 512x512 1 runway, 12.2ms
image 5/276 C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\generation_01\KATL_8L\footage\KATL_8L_004.jpg: 512x512 1 runway, 13.3ms
image 6/276 C:\Users\Sr-